# 02 — Nelson-Siegel Decomposition

Decompose each region's yield curve into **Level**, **Slope**, and **Curvature** factors
using the Diebold-Li (2006) approach with fixed $\lambda$.

$$y(\tau) = \beta_1 + \beta_2 \frac{1-e^{-\lambda\tau}}{\lambda\tau} + \beta_3 \left(\frac{1-e^{-\lambda\tau}}{\lambda\tau} - e^{-\lambda\tau}\right)$$

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.nelson_siegel import (
    ns_factor_loadings,
    fit_ns_ols,
    fit_ns_nonlinear,
    compute_fitted_yields,
    compute_rmse,
    select_lambda,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

MATURITIES = np.array([1, 2, 3, 5, 10, 20], dtype=float)

## 1. Load Aligned Yield Data

In [ ]:
combined = pd.read_csv('../data/raw/aligned_yields.csv', index_col=0, parse_dates=True)
print(f"Shape: {combined.shape}, Date range: {combined.index.min()} to {combined.index.max()}")

# Separate by region
uk_yields = combined[[c for c in combined.columns if c.startswith('UK_')]]
us_yields = combined[[c for c in combined.columns if c.startswith('US_')]]
eu_yields = combined[[c for c in combined.columns if c.startswith('EU_')]]

print(f"UK: {uk_yields.shape}, US: {us_yields.shape}, EU: {eu_yields.shape}")

## 2. Factor Loadings Visualization

Show how each factor loads on different maturities for $\lambda = 0.0609$.

In [ ]:
tau_fine = np.linspace(0.25, 30, 200)
X_fine = ns_factor_loadings(tau_fine, lam=0.0609)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tau_fine, X_fine[:, 0], label='Level ($\\beta_1$)', lw=2)
ax.plot(tau_fine, X_fine[:, 1], label='Slope ($\\beta_2$)', lw=2)
ax.plot(tau_fine, X_fine[:, 2], label='Curvature ($\\beta_3$)', lw=2)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Factor Loading')
ax.set_title('Nelson-Siegel Factor Loadings ($\\lambda = 0.0609$)')
ax.legend()
ax.axhline(0, color='grey', lw=0.5)
plt.tight_layout()
plt.show()

## 3. Lambda Selection

Test several $\lambda$ values and pick by average RMSE.

In [ ]:
# Use UK yields for lambda calibration
lambda_result = select_lambda(uk_yields, MATURITIES)
print(f"Best lambda: {lambda_result['best_lambda']:.4f}")
print("\nRMSE by lambda:")
for lam, rmse in sorted(lambda_result['rmse_by_lambda'].items()):
    marker = ' <-- best' if lam == lambda_result['best_lambda'] else ''
    print(f"  λ = {lam:.4f}: RMSE = {rmse:.4f} bps{marker}")

LAMBDA = lambda_result['best_lambda']

## 4. Fit Nelson-Siegel Factors (All Regions)

In [ ]:
uk_factors = fit_ns_ols(uk_yields, MATURITIES, lam=LAMBDA)
us_factors = fit_ns_ols(us_yields, MATURITIES, lam=LAMBDA)
eu_factors = fit_ns_ols(eu_yields, MATURITIES, lam=LAMBDA)

print(f"UK factors: {uk_factors.shape} — NaN: {uk_factors.isna().sum().sum()}")
print(f"US factors: {us_factors.shape} — NaN: {us_factors.isna().sum().sum()}")
print(f"EU factors: {eu_factors.shape} — NaN: {eu_factors.isna().sum().sum()}")

## 5. Goodness of Fit — Fitted vs Actual

In [ ]:
# Compute RMSE for UK
uk_rmse = compute_rmse(uk_yields, uk_factors, MATURITIES, lam=LAMBDA)
print(f"UK mean RMSE: {uk_rmse.mean():.4f}, median: {uk_rmse.median():.4f}, max: {uk_rmse.max():.4f}")

# Plot fitted vs actual for a sample date
sample_date = uk_yields.dropna().index[len(uk_yields.dropna()) // 2]
actual = uk_yields.loc[sample_date].values
fitted = compute_fitted_yields(
    uk_factors.loc[[sample_date]], MATURITIES, lam=LAMBDA
).values.flatten()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(MATURITIES, actual, 'o-', label='Actual', markersize=8)
ax.plot(MATURITIES, fitted, 's--', label='NS Fitted', markersize=8)
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.set_title(f'UK Yield Curve Fit — {sample_date.strftime("%Y-%m-%d")}')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Factor Time Series (All Regions)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for i, factor in enumerate(['Level', 'Slope', 'Curvature']):
    ax = axes[i]
    ax.plot(uk_factors.index, uk_factors[factor], label='UK', alpha=0.8)
    ax.plot(us_factors.index, us_factors[factor], label='US', alpha=0.8)
    ax.plot(eu_factors.index, eu_factors[factor], label='EU', alpha=0.8)
    ax.set_ylabel(factor)
    ax.set_title(f'{factor} Factor')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 7. Cross-Check: OLS vs Nonlinear Fit (Single Date)

In [ ]:
# Compare fixed-lambda OLS vs estimated-lambda for a single date
sample_yields = uk_yields.loc[sample_date].values
nl_result = fit_ns_nonlinear(sample_yields, MATURITIES)

print(f"Date: {sample_date.strftime('%Y-%m-%d')}")
print(f"\nOLS (λ={LAMBDA:.4f}):")
print(f"  Level={uk_factors.loc[sample_date, 'Level']:.4f}, "
      f"Slope={uk_factors.loc[sample_date, 'Slope']:.4f}, "
      f"Curvature={uk_factors.loc[sample_date, 'Curvature']:.4f}")
print(f"\nNonlinear (λ={nl_result['lambda']:.4f}):")
print(f"  Level={nl_result['Level']:.4f}, "
      f"Slope={nl_result['Slope']:.4f}, "
      f"Curvature={nl_result['Curvature']:.4f}")
print(f"  RMSE={nl_result['rmse']:.4f}")

## 8. Save Factors

In [ ]:
# Combine all factors into one DataFrame
all_factors = pd.concat([
    uk_factors.add_prefix('UK_'),
    us_factors.add_prefix('US_'),
    eu_factors.add_prefix('EU_'),
], axis=1)

all_factors.to_csv('../data/factors/ns_factors.csv')
print(f"Saved NS factors to data/factors/ns_factors.csv — shape {all_factors.shape}")
all_factors.describe()